# Evaluasi Dialect Normalization

Notebook ini membandingkan dialect normalization `off` dan `on` untuk query `Q03` dari `Ground truth - System.csv` menggunakan `NDCG@30`.

Keduanya memakai collection `COLLECTION_HYBRID_M8_EF400`. Spell checker dimatikan di kedua skenario agar efek yang diukur benar-benar berasal dari `localspelling.convert_spelling`, bukan koreksi spellchecker.

In [2]:
from __future__ import annotations

import importlib
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from qdrant_client import models
from qdrant_client.models import FieldCondition, Filter, MatchValue

try:
    from IPython.display import display
except ImportError:
    display = print

CWD = Path.cwd().resolve()
if (CWD / "Ground truth - System.csv").exists():
    EVAL_DIR = CWD
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD
    EVAL_DIR = PROJECT_ROOT / "evaluation"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import src.config as config_module
import src.search_engine as search_engine_module

config_module = importlib.reload(config_module)
search_engine_module = importlib.reload(search_engine_module)
settings = config_module.settings
SearchService = search_engine_module.SearchService
localspelling = search_engine_module.localspelling

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_DIR:", EVAL_DIR)
print("Config file:", config_module.__file__)
print("Search engine file:", search_engine_module.__file__)

PROJECT_ROOT: /home/vinik/Github/vector-search-thesis
EVAL_DIR: /home/vinik/Github/vector-search-thesis/evaluation
Config file: /home/vinik/Github/vector-search-thesis/src/config.py
Search engine file: /home/vinik/Github/vector-search-thesis/src/search_engine.py


In [3]:
GT_PATH = EVAL_DIR / "Ground truth - System.csv"
OUTPUT_DIR = EVAL_DIR / "dialect_eval_results"

QUERY_ID = "Q03"
K = 30
SEARCH_HNSW_EF = 256
RAW_QDRANT_LIMIT = 100
FILTER_TO_GT_POOL = True

COLLECTION_CONFIG = {
    "collection_label": "COLLECTION_HYBRID_M8_EF400_RAW",
    "collection_name": settings.COLLECTION_HYBRID_M8_EF400_RAW,
    "m": 8,
    "ef_construct": 400,
}

SCENARIO_CONFIGS = [
    {
        "scenario": "dialect_off",
        "use_dialect_normalization": False,
        "use_spellchecker": False,
        "description": "Query UK spelling asli tanpa localspelling.",
    },
    {
        "scenario": "dialect_on",
        "use_dialect_normalization": True,
        "use_spellchecker": False,
        "description": "Query UK spelling dinormalisasi ke US spelling dengan localspelling.",
    },
]

display(pd.DataFrame([COLLECTION_CONFIG]))
display(pd.DataFrame(SCENARIO_CONFIGS))

,collection_label,collection_name,m,ef_construct
0,COLLECTION_HYBRID_M8_EF400_RAW,rtas_hybrid_m8_ef400_raw,8,400


,scenario,use_dialect_normalization,use_spellchecker,description
0,dialect_off,False,False,Query UK spelling asli tanpa localspelling.
1,dialect_on,True,False,Query UK spelling dinormalisasi ke US spelling...


## Load Ground Truth Q03

In [4]:
def load_ground_truth_for_query(path: Path, query_id: str) -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(path)
    required_columns = {
        "query",
        "query_id",
        "rta_id",
        "agreement_title",
        "chapter_title",
        "article_number",
        "article_title",
        "article_text",
        "relevance",
    }
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing GT columns: {sorted(missing)}")

    query_mask = df["query_id"].astype(str).str.casefold().eq(query_id.casefold())
    query_df = df.loc[query_mask].copy()
    if query_df.empty:
        raise ValueError(f"Tidak ada ground truth untuk query_id={query_id!r}")

    query_df["relevance"] = pd.to_numeric(query_df["relevance"], errors="coerce").fillna(0).astype(float)
    query_values = query_df["query"].dropna().astype(str).unique().tolist()
    if len(query_values) != 1:
        raise ValueError(f"query_id={query_id!r} memiliki lebih dari satu query: {query_values}")

    return query_df.reset_index(drop=True), query_values[0]


gt_df, QUERY_TEXT = load_ground_truth_for_query(GT_PATH, QUERY_ID)

print(f"Query ID: {QUERY_ID}")
print(f"Query: {QUERY_TEXT}")
print(f"GT rows: {len(gt_df)}")
print("Dialect-normalized query:", localspelling.convert_spelling(QUERY_TEXT, "us"))
display(gt_df["relevance"].value_counts().sort_index().rename_axis("relevance").reset_index(name="count"))
display(
    gt_df[
        [
            "query_id",
            "query",
            "rta_id",
            "agreement_title",
            "article_number",
            "article_title",
            "relevance",
        ]
    ].head(10)
)

Query ID: Q03
Query: labour standards in trade
GT rows: 28
Dialect-normalized query: labor standards in trade


,relevance,count
0,0.0,9
1,1.0,1
2,2.0,7
3,3.0,11


,query_id,query,rta_id,agreement_title,article_number,article_title,relevance
0,Q03,labour standards in trade,174.0,Peru - Australia,Article 17.4,Understandings in Respect of this Chapter,3.0
1,Q03,labour standards in trade,NaN,United Kingdom - Canada,Article 23.3,Multilateral labour standards and agreements,3.0
2,Q03,labour standards in trade,NaN,United Kingdom - Canada,Article 23.2,Right to regulate and levels of protection,2.0
3,Q03,labour standards in trade,NaN,United Kingdom - Canada,Article 23.1,Context and objectives,2.0
4,Q03,labour standards in trade,NaN,United Kingdom - New Zealand FTA,Article 23.1,Definitions,2.0
5,Q03,labour standards in trade,NaN,United Kingdom - New Zealand FTA,Article 23.6,Trade and Labour,3.0
6,Q03,labour standards in trade,NaN,United Kingdom - New Zealand FTA,Article 23.4,Right to Regulate and Levels of Protection,2.0
7,Q03,labour standards in trade,NaN,United Kingdom - India,Article 20.3,Trade and Labour,3.0
8,Q03,labour standards in trade,NaN,United Kingdom - India,Article 20.2,Statement of Shared Commitment,2.0
9,Q03,labour standards in trade,NaN,United Kingdom - India,Article 20.10,Public Submissions,1.0


## Helper Normalisasi dan NDCG

In [5]:
def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\u00e2\u0080\u0099", "'").replace("\u2019", "'").replace("\u00a0", " ")
    text = re.sub(r"\bartical\b", "article", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip().casefold()
    return text


def normalize_rta_id(value: Any) -> str:
    if pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()


def article_match_keys(
    rta_id: Any,
    agreement_title: Any,
    article_number: Any,
    article_title: Any,
) -> list[tuple[str, ...]]:
    rid = normalize_rta_id(rta_id)
    agreement = normalize_text(agreement_title)
    number = normalize_text(article_number)
    title = normalize_text(article_title)

    keys: list[tuple[str, ...]] = []
    if rid and number and title:
        keys.append(("rta_number_title", rid, number, title))
    if rid and title:
        keys.append(("rta_title", rid, title))
    if agreement and number and title:
        keys.append(("agreement_number_title", agreement, number, title))
    if agreement and title:
        keys.append(("agreement_title", agreement, title))
    return keys


def build_gt_lookup(gt: pd.DataFrame) -> dict[tuple[str, ...], dict[str, Any]]:
    lookup: dict[tuple[str, ...], dict[str, Any]] = {}
    for gt_index, row in gt.iterrows():
        record = {
            "gt_index": gt_index,
            "relevance": float(row["relevance"]),
            "rta_id": row.get("rta_id"),
            "agreement_title": row.get("agreement_title"),
            "chapter_title": row.get("chapter_title"),
            "article_number": row.get("article_number"),
            "article_title": row.get("article_title"),
        }
        for key in article_match_keys(
            row.get("rta_id"),
            row.get("agreement_title"),
            row.get("article_number"),
            row.get("article_title"),
        ):
            if key not in lookup or record["relevance"] > lookup[key]["relevance"]:
                lookup[key] = record
    return lookup


def prediction_match_keys(prediction: dict[str, Any]) -> list[tuple[str, ...]]:
    return article_match_keys(
        prediction.get("rta_id"),
        prediction.get("agreement"),
        prediction.get("article_number"),
        prediction.get("article_name"),
    )


def find_prediction_in_gt(
    prediction: dict[str, Any],
    gt_lookup: dict[tuple[str, ...], dict[str, Any]],
) -> tuple[tuple[str, ...] | None, dict[str, Any] | None]:
    for key in prediction_match_keys(prediction):
        if key in gt_lookup:
            return key, gt_lookup[key]
    return None, None


def compute_dcg(scores: list[float]) -> float:
    return sum(float(rel) / np.log2(idx + 2) for idx, rel in enumerate(scores))


def compute_metric_bundle(predicted_relevance: list[float], ideal_relevance: list[float], k: int = K) -> dict[str, float]:
    scores = list(predicted_relevance[:k])
    if len(scores) < k:
        scores.extend([0.0] * (k - len(scores)))

    ideal = sorted([float(rel) for rel in ideal_relevance], reverse=True)[:k]
    if len(ideal) < k:
        ideal.extend([0.0] * (k - len(ideal)))

    dcg = compute_dcg(scores)
    idcg = compute_dcg(ideal)
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return {"dcg": dcg, "idcg": idcg, "ndcg": ndcg}

## Retrieval Qdrant dan Evaluasi Dialect

In [ ]:
engine = SearchService(
    use_reranker=False,
    use_llm=False,
    use_finetuned=False,
    use_dialect_normalization=False,
    use_spellchecker=False,
)


def prepare_query(engine: SearchService, query: str) -> tuple[str, Any]:
    search_query = query
    detected_iso = engine.detect_iso_from_query(search_query)
    qdrant_filter = None
    if detected_iso:
        qdrant_filter = Filter(
            must=[FieldCondition(key="iso", match=MatchValue(value=detected_iso))]
        )

    if engine.use_spellchecker and not engine.use_finetuned:
        words = search_query.split()
        misspelled = engine.spell.unknown(words)
        corrected = []
        for word in words:
            if word in misspelled:
                correction = engine.spell.correction(word)
                corrected.append(correction if correction else word)
            else:
                corrected.append(word)
        search_query = " ".join(corrected)

    if engine.use_dialect_normalization:
        uk_query = search_query
        us_query = localspelling.convert_spelling(search_query, "us")
        
        # Encode keduanya
        vec_uk = engine.vector_model.encode(uk_query)
        vec_us = engine.vector_model.encode(us_query)
        
        # Rata-ratakan vektornya (Vector Averaging)
        query_vector = ((vec_uk + vec_us) / 2).tolist()
        final_query = query_vector
    else:
        final_query = search_query

    return final_query, qdrant_filter


def resolve_article_text_display(payload: dict[str, Any]) -> str:
    article_text = payload.get("article_text") or "-"
    display_text = payload.get("article_text_display")
    if display_text is None:
        return article_text

    display_text = str(display_text).strip()
    if not display_text or display_text == "-" or display_text.lower().startswith("http") or display_text.lower().startswith("<a"):
        return article_text
    return display_text


def build_prediction_from_hit(hit: Any, rank: int) -> dict[str, Any]:
    payload = hit.payload or {}
    return {
        "rank": rank,
        "point_id": hit.id,
        "score": float(hit.score),
        "rta_id": payload.get("rta_id"),
        "agreement": payload.get("Agreement") or "-",
        "chapter_name": payload.get("chapter_name") or "-",
        "article_id": payload.get("article_id"),
        "article_number": payload.get("article_number") or "-",
        "article_name": payload.get("article_name") or "-",
        "article_text": payload.get("article_text") or "-",
        "article_text_display": resolve_article_text_display(payload),
        "chunk_index": payload.get("chunk_index"),
        "total_chunks": payload.get("total_chunks"),
        "chunk_method": payload.get("chunk_method"),
        "chunk_tokens": payload.get("chunk_tokens"),
    }


def dedupe_hits_to_articles(hits: list[Any]) -> list[Any]:
    unique_hits = []
    seen_article_keys = set()

    for hit in hits:
        payload = hit.payload or {}
        keys = article_match_keys(
            payload.get("rta_id"),
            payload.get("Agreement"),
            payload.get("article_number"),
            payload.get("article_name"),
        )
        dedupe_key = keys[0] if keys else ("point_id", str(hit.id))
        if dedupe_key in seen_article_keys:
            continue
        seen_article_keys.add(dedupe_key)
        unique_hits.append(hit)

    return unique_hits


def retrieve_articles(
    engine: SearchService,
    query: str,
    collection_name: str,
    raw_limit: int = RAW_QDRANT_LIMIT,
    hnsw_ef: int = SEARCH_HNSW_EF,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    final_query, qdrant_filter = prepare_query(engine, query)
    query_vector = engine.vector_model.encode(final_query).tolist()

    started = time.perf_counter()
    response = engine.client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=qdrant_filter,
        limit=raw_limit,
        search_params=models.SearchParams(hnsw_ef=hnsw_ef),
        with_payload=True,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000

    raw_hits = response.points
    unique_hits = dedupe_hits_to_articles(raw_hits)
    predictions = [build_prediction_from_hit(hit, rank=idx) for idx, hit in enumerate(unique_hits, start=1)]
    metadata = {
        "final_query": final_query,
        "raw_hit_count": len(raw_hits),
        "unique_article_count": len(unique_hits),
        "elapsed_ms": elapsed_ms,
    }
    return predictions, metadata


def evaluate_predictions(
    predictions: list[dict[str, Any]],
    gt: pd.DataFrame,
    k: int = K,
    filter_to_gt_pool: bool = FILTER_TO_GT_POOL,
) -> tuple[dict[str, float], list[float], list[dict[str, Any]]]:
    gt_lookup = build_gt_lookup(gt)
    relevance_sequence: list[float] = []
    detail_rows: list[dict[str, Any]] = []
    seen_gt_keys = set()

    for prediction in predictions:
        gt_key, gt_record = find_prediction_in_gt(prediction, gt_lookup)

        if gt_record is None:
            if filter_to_gt_pool:
                continue
            relevance = 0.0
        else:
            if gt_key in seen_gt_keys:
                continue
            seen_gt_keys.add(gt_key)
            relevance = float(gt_record["relevance"])

        relevance_sequence.append(relevance)
        detail_rows.append(
            {
                "rank_after_eval_filter": len(relevance_sequence),
                "rank_before_eval_filter": prediction.get("rank"),
                "score": prediction.get("score"),
                "relevance": relevance,
                "matched_in_gt": gt_record is not None,
                "rta_id": prediction.get("rta_id"),
                "agreement": prediction.get("agreement"),
                "chapter_name": prediction.get("chapter_name"),
                "article_number": prediction.get("article_number"),
                "article_name": prediction.get("article_name"),
                "chunk_index": prediction.get("chunk_index"),
                "total_chunks": prediction.get("total_chunks"),
                "chunk_method": prediction.get("chunk_method"),
                "chunk_tokens": prediction.get("chunk_tokens"),
                "gt_rta_id": None if gt_record is None else gt_record.get("rta_id"),
                "gt_agreement_title": None if gt_record is None else gt_record.get("agreement_title"),
                "gt_article_number": None if gt_record is None else gt_record.get("article_number"),
                "gt_article_title": None if gt_record is None else gt_record.get("article_title"),
            }
        )

        if len(relevance_sequence) >= k:
            break

    metrics = compute_metric_bundle(relevance_sequence, gt["relevance"].tolist(), k=k)
    return metrics, relevance_sequence, detail_rows


def evaluate_dialect_scenarios(
    scenario_configs: list[dict[str, Any]],
    gt: pd.DataFrame,
    query: str,
    collection_config: dict[str, Any] = COLLECTION_CONFIG,
    k: int = K,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    detail_rows = []

    for scenario_config in scenario_configs:
        scenario = scenario_config["scenario"]
        engine.use_dialect_normalization = bool(scenario_config["use_dialect_normalization"])
        engine.use_spellchecker = bool(scenario_config["use_spellchecker"])
        if not engine.use_spellchecker:
            engine.spell = None

        print(f"Evaluating {scenario}")
        predictions, metadata = retrieve_articles(
            engine=engine,
            query=query,
            collection_name=collection_config["collection_name"],
            raw_limit=RAW_QDRANT_LIMIT,
            hnsw_ef=SEARCH_HNSW_EF,
        )
        metrics, relevance_sequence, rows = evaluate_predictions(
            predictions=predictions,
            gt=gt,
            k=k,
            filter_to_gt_pool=FILTER_TO_GT_POOL,
        )

        summary_rows.append(
            {
                "query_id": QUERY_ID,
                "query": query,
                "scenario": scenario,
                "use_dialect_normalization": bool(scenario_config["use_dialect_normalization"]),
                "use_spellchecker": bool(scenario_config["use_spellchecker"]),
                "collection_label": collection_config["collection_label"],
                "collection_name": collection_config["collection_name"],
                "m": collection_config["m"],
                "ef_construct": collection_config["ef_construct"],
                "search_hnsw_ef": SEARCH_HNSW_EF,
                "raw_qdrant_limit": RAW_QDRANT_LIMIT,
                "final_query": metadata["final_query"],
                "gt_rows": len(gt),
                "raw_hit_count": metadata["raw_hit_count"],
                "unique_article_count": metadata["unique_article_count"],
                "evaluated_rows": len(relevance_sequence),
                "matched_gt_rows_at_k": sum(1 for row in rows if row["matched_in_gt"]),
                f"dcg_at_{k}": metrics["dcg"],
                f"idcg_at_{k}": metrics["idcg"],
                f"ndcg_at_{k}": metrics["ndcg"],
                "elapsed_ms": metadata["elapsed_ms"],
                "relevance_sequence": relevance_sequence,
            }
        )

        for row in rows:
            detail_rows.append(
                {
                    "query_id": QUERY_ID,
                    "query": query,
                    "scenario": scenario,
                    "use_dialect_normalization": bool(scenario_config["use_dialect_normalization"]),
                    "collection_label": collection_config["collection_label"],
                    "collection_name": collection_config["collection_name"],
                    **row,
                }
            )

    summary_df = pd.DataFrame(summary_rows).sort_values(f"ndcg_at_{k}", ascending=False).reset_index(drop=True)
    detail_df = pd.DataFrame(detail_rows)
    return summary_df, detail_df

2026-06-03 21:20:44 - src.search_engine - INFO - [Init] Initializing model & database
2026-06-03 21:20:44 - httpx - INFO - HTTP Request: GET http://127.0.0.1:6333 "HTTP/1.1 200 OK"
2026-06-03 21:20:44 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2026-06-03 21:20:44 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2
2026-06-03 21:20:44 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-03 21:20:44 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
2026-06-03 21:20:44 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary R

## Jalankan Evaluasi

In [7]:
dialect_summary_df, dialect_detail_df = evaluate_dialect_scenarios(
    scenario_configs=SCENARIO_CONFIGS,
    gt=gt_df,
    query=QUERY_TEXT,
    collection_config=COLLECTION_CONFIG,
    k=K,
)

summary_cols = [
    "query_id",
    "scenario",
    "use_dialect_normalization",
    "use_spellchecker",
    "collection_label",
    "collection_name",
    "search_hnsw_ef",
    "raw_qdrant_limit",
    "final_query",
    "raw_hit_count",
    "unique_article_count",
    "evaluated_rows",
    "matched_gt_rows_at_k",
    f"dcg_at_{K}",
    f"idcg_at_{K}",
    f"ndcg_at_{K}",
    "elapsed_ms",
    "relevance_sequence",
]

detail_cols = [
    "scenario",
    "rank_after_eval_filter",
    "rank_before_eval_filter",
    "score",
    "relevance",
    "matched_in_gt",
    "rta_id",
    "agreement",
    "article_number",
    "article_name",
    "chunk_index",
    "total_chunks",
    "chunk_method",
    "chunk_tokens",
]

display(dialect_summary_df[summary_cols])
if dialect_detail_df.empty:
    display(pd.DataFrame(columns=detail_cols))
else:
    display(dialect_detail_df[detail_cols].head(K * len(SCENARIO_CONFIGS)))

Evaluating dialect_off


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]
2026-06-03 21:20:51 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef400_raw/points/query "HTTP/1.1 200 OK"


Evaluating dialect_on


Batches: 100%|██████████| 1/1 [00:00<00:00, 133.97it/s]
2026-06-03 21:20:51 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef400_raw/points/query "HTTP/1.1 200 OK"


,query_id,scenario,use_dialect_normalization,use_spellchecker,collection_label,collection_name,search_hnsw_ef,raw_qdrant_limit,final_query,raw_hit_count,unique_article_count,evaluated_rows,matched_gt_rows_at_k,dcg_at_30,idcg_at_30,ndcg_at_30,elapsed_ms,relevance_sequence
0,Q03,dialect_off,False,False,COLLECTION_HYBRID_M8_EF400_RAW,rtas_hybrid_m8_ef400_raw,256,100,labour standards in trade,100,94,20,20,13.645514,18.21632,0.749082,322.120946,"[0.0, 3.0, 3.0, 2.0, 2.0, 3.0, 3.0, 2.0, 0.0, ..."
1,Q03,dialect_on,True,False,COLLECTION_HYBRID_M8_EF400_RAW,rtas_hybrid_m8_ef400_raw,256,100,labor standards in trade,100,97,17,17,10.838617,18.21632,0.594995,33.788016,"[0.0, 2.0, 0.0, 2.0, 3.0, 3.0, 3.0, 2.0, 3.0, ..."


,scenario,rank_after_eval_filter,rank_before_eval_filter,score,relevance,matched_in_gt,rta_id,agreement,article_number,article_name,chunk_index,total_chunks,chunk_method,chunk_tokens
0,dialect_off,1,2,0.706527,0.0,True,249,European Free Trade Association (EFTA),-,Preamble,1,2,Recursive-Fallback,97
1,dialect_off,2,3,0.691690,3.0,True,583,United Kingdom - Canada,Article 23.3,Multilateral labour standards and agreements,0,2,Recursive-Fallback,380
2,dialect_off,3,4,0.689957,3.0,True,445,United Kingdom - Singapore,ARTICLE 12.3,Multilateral Labour Standards and Agreements,0,2,Recursive-Fallback,380
3,dialect_off,4,6,0.686938,2.0,True,510,United Kingdom - New Zealand FTA,Article 23.1,Definitions,0,1,Semantic,331
4,dialect_off,5,7,0.677258,2.0,True,583,United Kingdom - Canada,Article 23.1,Context and objectives,0,1,Semantic,166
5,dialect_off,6,8,0.676613,3.0,True,447,EU - Canada,ARTICAL 23.3,Multilateral labour standards and agreements,0,2,Recursive-Fallback,380
6,dialect_off,7,10,0.673668,3.0,True,479,"United Kingdom - Korea, Republic of",ARTICLE 13.2,Scope,0,1,Semantic,82
7,dialect_off,8,12,0.668780,2.0,True,583,United Kingdom - Canada,Article 23.2,Right to regulate and levels of protection,0,1,Semantic,104
8,dialect_off,9,13,0.668508,0.0,True,10005,AGREEMENT ON TECHNICAL BARRIERS TO TRADE,Article 4,"Preparation, Adoption and Application of Stand...",0,1,Semantic,227
9,dialect_off,10,18,0.663040,3.0,True,608,United Kingdom - India,Article 20.3,Trade and Labour,1,2,Recursive-Fallback,178


In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / f"dialect_{QUERY_ID.lower()}_summary_ndcg_at_{K}.csv"
detail_path = OUTPUT_DIR / f"dialect_{QUERY_ID.lower()}_detail_ndcg_at_{K}.csv"

dialect_summary_df.to_csv(summary_path, index=False)
dialect_detail_df.to_csv(detail_path, index=False)

print("Saved outputs:")
print("-", summary_path)
print("-", detail_path)

Saved outputs:
- /home/vinik/Github/vector-search-thesis/evaluation/dialect_eval_results/dialect_q03_summary_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/dialect_eval_results/dialect_q03_detail_ndcg_at_30.csv
